# Single Best Next Skill Prediction using XGBoost

This notebook implements a model to predict the single most important next skill to learn given:
- Current IT skills
- Current soft skills  
- Target profession

Unlike the LSTM model that predicts multiple skills, this focuses on the ONE best next skill.

## Imports

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import xgboost as xgb
import pickle
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

print(f"XGBoost version: {xgb.__version__}")

## Load Training Data

In [ ]:
# Load main training data
with open("./data/processed/training_data.json", "r") as f:
    training_data = json.load(f)
    
# Load profession-specific skill weights
with open("./data/processed/designation_aggregated_skills.json", "r") as f:
    profession_skills = json.load(f)

print(f"Loaded {len(training_data)} training examples")
print(f"Loaded {len(profession_skills)} profession skill mappings")

## Convert Multi-label to Single-label Problem

We convert the multi-skill prediction problem to single-skill by selecting the highest weighted next skill per example.

In [ ]:
def prepare_single_skill_dataset(training_data, profession_skills):
    """
    Convert multi-label problem to single-label by selecting the best next skill per example
    """
    print("Preparing single-skill dataset...")
    
    examples = []
    all_skills = set()
    all_soft_skills = set()
    all_professions = set()
    
    for i, example in enumerate(training_data):
        if i % 10000 == 0:
            print(f"Processed {i} examples...")
            
        # Extract features
        it_skills = set(example.get("it_skill_categories", []))
        soft_skills = set(example.get("soft_skills", []))
        profession = example.get("desired_designation", "")
        
        # Get next skills with counts
        next_it_skills = example.get("next_skill", {})
        next_soft_skills = example.get("next_soft_skill", {})
        
        # Skip if no next skills
        if not next_it_skills and not next_soft_skills:
            continue
            
        # Combine all next skills and select the best one
        all_next_skills = {}
        
        # Add IT skills with profession weighting
        prof_it_weights = profession_skills.get(profession, {}).get("it_skills", {})
        for skill, count in next_it_skills.items():
            prof_weight = prof_it_weights.get(skill, 1)
            weighted_score = count * prof_weight
            all_next_skills[f"IT_{skill}"] = weighted_score
            
        # Add soft skills with profession weighting  
        prof_soft_weights = profession_skills.get(profession, {}).get("soft_skills", {})
        for skill, count in next_soft_skills.items():
            prof_weight = prof_soft_weights.get(skill, 1)
            weighted_score = count * prof_weight
            all_next_skills[f"SOFT_{skill}"] = weighted_score
        
        # Select best next skill
        if all_next_skills:
            best_skill = max(all_next_skills.keys(), key=lambda k: all_next_skills[k])
            
            # Store for feature collection
            all_skills.update(it_skills)
            all_soft_skills.update(soft_skills)
            all_professions.add(profession)
            
            examples.append({
                "it_skills": list(it_skills),
                "soft_skills": list(soft_skills), 
                "profession": profession,
                "target_skill": best_skill,
                "target_weight": all_next_skills[best_skill]
            })
    
    print(f"Created {len(examples)} single-skill training examples")
    print(f"Unique IT skills: {len(all_skills)}")
    print(f"Unique soft skills: {len(all_soft_skills)}")
    print(f"Unique professions: {len(all_professions)}")
    
    return examples, all_skills, all_soft_skills, all_professions

examples, all_skills, all_soft_skills, all_professions = prepare_single_skill_dataset(training_data, profession_skills)

## Analyze Target Distribution

In [ ]:
# Analyze target skill distribution
target_skills = [ex['target_skill'] for ex in examples]
target_counter = Counter(target_skills)

print(f"Total unique target skills: {len(target_counter)}")
print(f"\nTop 10 most common target skills:")
for skill, count in target_counter.most_common(10):
    print(f"  {skill}: {count}")

# Plot distribution
top_20_skills = target_counter.most_common(20)
skills, counts = zip(*top_20_skills)

plt.figure(figsize=(12, 8))
plt.barh(range(len(skills)), counts)
plt.yticks(range(len(skills)), [s.replace('_', ' ') for s in skills])
plt.xlabel('Frequency')
plt.title('Top 20 Most Common Target Skills')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Check IT vs Soft skill distribution
it_count = sum(1 for skill in target_skills if skill.startswith('IT_'))
soft_count = sum(1 for skill in target_skills if skill.startswith('SOFT_'))
print(f"\nTarget skill types:")
print(f"IT skills: {it_count} ({it_count/len(target_skills)*100:.1f}%)")
print(f"Soft skills: {soft_count} ({soft_count/len(target_skills)*100:.1f}%)")

## Feature Engineering

In [ ]:
def create_features(examples, all_skills, all_soft_skills, all_professions):
    """Create feature matrix for XGBoost"""
    print("Creating feature matrix...")
    
    features_list = []
    targets = []
    
    # Create feature columns list
    feature_columns = []
    
    # Binary features for current IT skills
    for skill in sorted(all_skills):
        feature_columns.append(f"has_it_{skill}")
        
    # Binary features for current soft skills
    for skill in sorted(all_soft_skills):
        feature_columns.append(f"has_soft_{skill}")
        
    # Categorical feature for profession (one-hot encoded)
    for profession in sorted(all_professions):
        feature_columns.append(f"prof_{profession}")
        
    # Count features
    feature_columns.extend([
        "it_skill_count", "soft_skill_count", "total_skill_count"
    ])
    
    # Create features for each example
    for example in examples:
        features = {}
        
        # IT skill binary features
        for skill in sorted(all_skills):
            features[f"has_it_{skill}"] = 1 if skill in example["it_skills"] else 0
            
        # Soft skill binary features  
        for skill in sorted(all_soft_skills):
            features[f"has_soft_{skill}"] = 1 if skill in example["soft_skills"] else 0
            
        # Profession one-hot encoding
        for profession in sorted(all_professions):
            features[f"prof_{profession}"] = 1 if profession == example["profession"] else 0
            
        # Count features
        features["it_skill_count"] = len(example["it_skills"])
        features["soft_skill_count"] = len(example["soft_skills"])
        features["total_skill_count"] = len(example["it_skills"]) + len(example["soft_skills"])
        
        features_list.append(features)
        targets.append(example["target_skill"])
    
    # Convert to DataFrame
    df_features = pd.DataFrame(features_list)
    
    # Ensure all columns exist
    for col in feature_columns:
        if col not in df_features.columns:
            df_features[col] = 0
            
    # Reorder columns
    df_features = df_features[feature_columns]
    
    print(f"Feature matrix shape: {df_features.shape}")
    print(f"Unique target skills: {len(set(targets))}")
    
    return df_features, targets, feature_columns

X, y, feature_columns = create_features(examples, all_skills, all_soft_skills, all_professions)

## Add Profession-Skill Interaction Features

In [ ]:
def add_interaction_features(df_features):
    """Add profession-skill interaction features"""
    print("Adding interaction features...")
    
    original_cols = df_features.columns.tolist()
    
    # Get profession and skill columns
    prof_cols = [col for col in df_features.columns if col.startswith("prof_")]
    it_cols = [col for col in df_features.columns if col.startswith("has_it_")]
    
    # Add top profession-IT skill interactions (limit to reduce complexity)
    # Select top 10 most common professions and top 20 IT skills
    prof_frequencies = df_features[prof_cols].sum().sort_values(ascending=False)
    it_frequencies = df_features[it_cols].sum().sort_values(ascending=False)
    
    top_prof_cols = prof_frequencies.head(10).index.tolist()
    top_it_cols = it_frequencies.head(20).index.tolist()
    
    print(f"Adding interactions for {len(top_prof_cols)} professions and {len(top_it_cols)} IT skills")
    
    for prof_col in top_prof_cols:
        for it_col in top_it_cols:
            interaction_name = f"int_{prof_col}_{it_col}"
            df_features[interaction_name] = df_features[prof_col] * df_features[it_col]
    
    print(f"Added {len(df_features.columns) - len(original_cols)} interaction features")
    print(f"Final feature matrix shape: {df_features.shape}")
    
    return df_features

X = add_interaction_features(X)

## Train-Test Split

In [ ]:
# Filter out rare target classes (appear less than 5 times)
target_counts = Counter(y)
valid_targets = [target for target, count in target_counts.items() if count >= 5]

# Filter dataset
valid_indices = [i for i, target in enumerate(y) if target in valid_targets]
X_filtered = X.iloc[valid_indices]
y_filtered = [y[i] for i in valid_indices]

print(f"Filtered dataset: {len(y_filtered)} examples with {len(set(y_filtered))} target classes")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_filtered, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered
)

print(f"Training set: {X_train.shape[0]} examples")
print(f"Test set: {X_test.shape[0]} examples")
print(f"Number of features: {X_train.shape[1]}")

## Train XGBoost Model

In [ ]:
# Encode target labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print(f"Number of target classes: {len(label_encoder.classes_)}")

# Configure XGBoost parameters
params = {
    'objective': 'multi:softprob',
    'num_class': len(label_encoder.classes_),
    'max_depth': 8,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': 42,
    'eval_metric': 'mlogloss'
}

# Create DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train_encoded)
dtest = xgb.DMatrix(X_test, label=y_test_encoded)

# Train with early stopping
evals = [(dtrain, 'train'), (dtest, 'test')]
model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=500,
    evals=evals,
    early_stopping_rounds=50,
    verbose_eval=50
)

## Evaluate Model

In [ ]:
# Make predictions
y_pred_proba = model.predict(dtest)
y_pred_encoded = np.argmax(y_pred_proba, axis=1)
y_pred = label_encoder.inverse_transform(y_pred_encoded)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

# Top-k accuracy
def top_k_accuracy(y_true, y_proba, k=3):
    """Calculate top-k accuracy"""
    correct = 0
    for i, true_label in enumerate(y_true):
        top_k_indices = np.argsort(y_proba[i])[-k:]
        top_k_labels = label_encoder.inverse_transform(top_k_indices)
        if true_label in top_k_labels:
            correct += 1
    return correct / len(y_true)

top3_acc = top_k_accuracy(y_test, y_pred_proba, k=3)
top5_acc = top_k_accuracy(y_test, y_pred_proba, k=5)

print(f"Top-3 Accuracy: {top3_acc:.4f}")
print(f"Top-5 Accuracy: {top5_acc:.4f}")

# Classification report for top classes
print("\nClassification Report (Top 10 most common classes):")
common_classes = [cls for cls, _ in Counter(y_test).most_common(10)]
y_test_filtered = [y for y in y_test if y in common_classes]
y_pred_filtered = [y_pred[i] for i, y in enumerate(y_test) if y in common_classes]

print(classification_report(y_test_filtered, y_pred_filtered, labels=common_classes, target_names=common_classes))

## Feature Importance Analysis

In [ ]:
# Get feature importance
importance = model.get_score(importance_type='weight')

# Map feature indices to names
feature_names = X_train.columns.tolist()
importance_named = {}
for i, col in enumerate(feature_names):
    key = f"f{i}"
    if key in importance:
        importance_named[col] = importance[key]

# Sort by importance
sorted_importance = sorted(importance_named.items(), key=lambda x: x[1], reverse=True)

print("Top 15 most important features:")
for i, (feature, score) in enumerate(sorted_importance[:15]):
    print(f"{i+1:2d}. {feature}: {score}")

# Plot feature importance
top_features = sorted_importance[:20]
features, scores = zip(*top_features)

plt.figure(figsize=(12, 8))
plt.barh(range(len(features)), scores)
plt.yticks(range(len(features)), [f.replace('_', ' ') for f in features])
plt.xlabel('Feature Importance Score')
plt.title('Top 20 Most Important Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Test Profession-Specific Predictions

In [ ]:
def predict_next_skill(current_it_skills, current_soft_skills, target_profession, top_k=3):
    """Predict next skill for given inputs"""
    
    # Create feature vector
    features = {}
    
    # IT skill features
    for skill in sorted(all_skills):
        features[f"has_it_{skill}"] = 1 if skill in current_it_skills else 0
        
    # Soft skill features  
    for skill in sorted(all_soft_skills):
        features[f"has_soft_{skill}"] = 1 if skill in current_soft_skills else 0
        
    # Profession features
    for profession in sorted(all_professions):
        features[f"prof_{profession}"] = 1 if profession == target_profession else 0
        
    # Count features
    features["it_skill_count"] = len(current_it_skills)
    features["soft_skill_count"] = len(current_soft_skills)
    features["total_skill_count"] = len(current_it_skills) + len(current_soft_skills)
    
    # Add interaction features (simplified)
    prof_col = f"prof_{target_profession}"
    for skill in current_it_skills:
        it_col = f"has_it_{skill}"
        interaction_name = f"int_{prof_col}_{it_col}"
        if interaction_name in X.columns:
            features[interaction_name] = 1
    
    # Create DataFrame with same columns as training data
    feature_df = pd.DataFrame([features])
    
    # Ensure all columns exist
    for col in X.columns:
        if col not in feature_df.columns:
            feature_df[col] = 0
            
    # Reorder columns to match training data
    feature_df = feature_df[X.columns]
    
    # Predict
    dtest_single = xgb.DMatrix(feature_df)
    probabilities = model.predict(dtest_single)[0]
    
    # Get top-k predictions
    top_indices = np.argsort(probabilities)[-top_k:][::-1]
    top_skills = label_encoder.inverse_transform(top_indices)
    top_probs = probabilities[top_indices]
    
    return list(zip(top_skills, top_probs))

# Test with different profession-skill combinations
test_cases = [
    {
        "it_skills": ["PYTHON PROGRAMMER", "DATA ANALYST"],
        "soft_skills": ["analytical thinking", "problem solving"],
        "profession": "DATA SCIENTIST"
    },
    {
        "it_skills": ["WEB DEVELOPER", "FRONTEND DEVELOPER"],
        "soft_skills": ["creativity", "teamwork"],
        "profession": "FRONTEND DEVELOPER"
    },
    {
        "it_skills": ["JAVA PROGRAMMER", "SOFTWARE DEVELOPER"],
        "soft_skills": ["leadership", "project management"],
        "profession": "SOFTWARE ARCHITECT"
    }
]

print("=== PROFESSION-SPECIFIC NEXT SKILL PREDICTIONS ===")
for i, case in enumerate(test_cases):
    print(f"\nTest Case {i+1}:")
    print(f"Current IT Skills: {case['it_skills']}")
    print(f"Current Soft Skills: {case['soft_skills']}")
    print(f"Target Profession: {case['profession']}")
    
    predictions = predict_next_skill(
        case['it_skills'], 
        case['soft_skills'], 
        case['profession'], 
        top_k=3
    )
    
    print("Top 3 Next Skill Predictions:")
    for j, (skill, prob) in enumerate(predictions):
        skill_clean = skill.replace('IT_', '').replace('SOFT_', '').replace('_', ' ')
        print(f"  {j+1}. {skill_clean} (confidence: {prob:.3f})")
    print("-" * 50)

## Save Model

In [ ]:
# Save the model and metadata
model_data = {
    'xgboost_model': model,
    'label_encoder': label_encoder,
    'feature_columns': X.columns.tolist(),
    'all_skills': list(all_skills),
    'all_soft_skills': list(all_soft_skills),
    'all_professions': list(all_professions),
    'model_params': params,
    'test_accuracy': accuracy,
    'top3_accuracy': top3_acc,
    'top5_accuracy': top5_acc
}

with open('../model/single_skill_xgboost.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print("Model saved successfully!")
print(f"Final model performance:")
print(f"- Test Accuracy: {accuracy:.4f}")
print(f"- Top-3 Accuracy: {top3_acc:.4f}")
print(f"- Top-5 Accuracy: {top5_acc:.4f}")
print(f"- Number of target classes: {len(label_encoder.classes_)}")
print(f"- Number of features: {X.shape[1]}")